# Reproducing Sohn et al. (2019) from dandiset 000130

Paper. Sohn H, Narain D, Meirhaeghe N, Jazayeri M (2019). Bayesian computation through cortical
latent dynamics. Neuron 103(5):934-947.e5.
[doi:10.1016/j.neuron.2019.06.012](https://doi.org/10.1016/j.neuron.2019.06.012). The claims
file (`claims/sohn-2019.yaml`) was written from the accepted author manuscript in PubMed Central
(PMC6805134): Results, figure legends and STAR Methods. The supplementary tables and figures were
not available to the reader. The paper's analysis code is available on request only.

Dataset. [DANDI:000130](https://dandiarchive.org/dandiset/000130/0.241017.1448), version
0.241017.1448 (doi:10.48324/dandi.000130/0.241017.1448). This version was asked for in the
prompt of this attempt; it is also the dandiset's only published version. The dandiset was
prepared for the Neural Latents Benchmark (DMFC_RSG) and holds one recording session
(2016-12-11) of monkey H ("Haydn"): a train file with the trial table and all 54 units, and a test
file with 1.7 s segments of 283 further trials for 40 units and no task variables. The paper
analysed 17 sessions of H and 12 of monkey G; only this session is here. Following the paper, a
"dataset" is one effector and target direction (4 here, 8 in the paper over both animals).

Data are streamed from the archive; nothing is written to disk. Loading and the measures are in
`rsg.py`, whose docstring says where each parameter comes from.

| Claim | Tested here |
|---|---|
| beh-slopes | tested |
| beh-overlap-prior | tested |
| beh-sigmoid-increments | tested |
| beh-long-more-bias | tested |
| beh-bayes-model | tested |
| glm-prior-ts | tested |
| dfr-overlap-support | tested |
| pca-dimensionality | tested |
| encoding-r2 | tested |
| encoding-bayes-beats-linear | tested |
| short-in-long-control | tested |
| neural-long-more-bias | tested |
| speed-vs-projection | tested |
| initial-speed-biased | tested |
| speed-predicts-tp | tested |
| alt-speed-unmodulated | tested |
| alt-transient-parallel | tested |
| alt-threshold-monotonic | tested |
| gpfa-variance-inverted-u | tested |
| gpfa-mean-sigmoid | tested |
| rnn-curved-manifold | blocked: the networks are not in the dataset (data_missing) |
| rnn-perturbation | blocked: the networks are not in the dataset (data_missing) |
| monkey-g | blocked: monkey G is not in the dataset (data_missing) |

In [ ]:
import logging
import resource
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy import stats

import rsg

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
T0 = time.time()
DS = list(rsg.DATASETS)
PRIORS = ["Short", "Long"]


def signed_rank(x, alternative="two-sided"):
    """Wilcoxon signed-rank test against zero, exact."""
    x = np.asarray(x, float)
    return stats.wilcoxon(x, alternative=alternative, method="exact").pvalue

In [ ]:
urls = rsg.asset_urls()
data = rsg.load_file(urls[rsg.TRAIN_PATH])
test = rsg.load_file(urls[rsg.TEST_PATH])
trials_all = data["trials"]
T = rsg.good_trials(trials_all)
keep, unit_table = rsg.select_units(data, T)
SP = [data["spikes"][i] for i in keep]
print(f"loaded in {time.time() - T0:.1f} s")

## Data check

Each check of the metadata against the values is stated under its output.

In [ ]:
print("Files at the pinned version:", sorted(urls))
for name, d in (("train", data), ("test", test)):
    print(f"\n{name}: {d['meta']}")
    print(f"  trials {len(d['trials'])}, columns {list(d['trials'].columns)}")
    print(f"  units {len(d['units'])} (held out in the benchmark: {int(d['units'].heldout.sum())})")
print("\nTrain-file trial columns and their descriptions:")
for c, desc in data["trial_descriptions"].items():
    print(f"  {c}: {desc}")
print("\nTrials by benchmark split (train file):", trials_all.split.value_counts().to_dict())
print("Test-file trial columns:", list(test["trials"].columns), "; split:",
      test["trials"].split.value_counts().to_dict())
done = trials_all.tp > 0
print(f"\nTrain-file trials: {len(trials_all)}; completed (tp > 0): {int(done.sum())}; "
      f"is_outlier: {int(trials_all.is_outlier.sum())} "
      f"(of which completed: {int((trials_all.is_outlier & done).sum())}); "
      f"kept (completed, not outlier): {len(T)}")
print("Rewarded among kept:", int(T.reward_time.notna().sum()), "; rewarded among outliers:",
      int(trials_all[trials_all.is_outlier].reward_time.notna().sum()))
print("\nKept trials per dataset, prior and ts:")
print(pd.crosstab([T.dataset, T.prior], T.ts).to_string())

In [ ]:
print("Ranges of the variables used (kept trials):")
for c in ["ts", "tp", "theta", "iti"]:
    print(f"  {c}: {T[c].min():.1f} to {T[c].max():.1f}")
print("  ts values:", sorted(T.ts.unique()))
order_ok = ((T.fix_time <= T.target_on_time) & (T.target_on_time <= T.ready_time)
            & (T.ready_time < T.set_time) & (T.set_time < T.go_time)).all()
print("Event order fix <= target <= ready < set < go in every kept trial:", bool(order_ok))
d_ts = (T.set_time - T.ready_time) * 1000 - T.ts
d_tp = (T.go_time - T.set_time) * 1000 - T.tp
print("(set_time - ready_time) - ts, ms: percentiles 0/1/50/99/100 =",
      np.percentile(d_ts, [0, 1, 50, 99, 100]).round(1))
print("(go_time - set_time) - tp, ms: percentiles 0/1/50/99/100 =",
      np.percentile(d_tp, [0, 1, 50, 99, 100]).round(1))
print("  by effector (median, ms):", d_tp.groupby(T.is_eye).median().round(1).to_dict())
print("\niti by prior (ms):")
print(pd.crosstab(trials_all.is_short.map({True: "Short", False: "Long"}), trials_all.iti).to_string())
print("\nis_short against ts range: Short ts", sorted(T[T.prior == "Short"].ts.unique()),
      "; Long ts", sorted(T[T.prior == "Long"].ts.unique()))

`ts` matches the time from `ready_time` to `set_time` to within about 24 ms (one or two frames of a
60 Hz display; the Methods give a 60 Hz monitor). `tp` is longer than `go_time - set_time` by a
median of about 18 ms, and the difference is not documented. Behaviour uses the `ts` and `tp`
columns (claims header) and spikes are aligned to the event times. The `iti` column is described
as 500 ms for the Short prior and 1220 ms for the Long prior; the values are the reverse (see the
table above), which matches the Methods ("we set the inter-trial intervals of the Short and Long
conditions to 1220 ms and 500 ms, respectively"). The description is wrong; `iti` is not used.

In [ ]:
print("Units in the train file:", len(data["units"]))
print("Observation intervals identical for all units:",
      all(np.array_equal(o, data["obs_intervals"][0]) for o in data["obs_intervals"]),
      "; covering", data["obs_intervals"][0].shape[0], "segments,",
      round(float(np.sum(np.diff(data["obs_intervals"][0], axis=1))), 1), "s")
print("Spike times sorted in every unit:", all(np.all(np.diff(s) >= 0) for s in data["spikes"]))
print("Spike time range, s:", round(min(s.min() for s in data["spikes"]), 3), "to",
      round(max(s.max() for s in data["spikes"]), 3), "; trials span",
      round(trials_all.start_time.min(), 3), "to", round(trials_all.stop_time.max(), 3))
print("Mean rates (spikes/s): min %.2f, median %.2f, max %.2f" % (
    unit_table.rate.min(), unit_table.rate.median(), unit_table.rate.max()))
print("Units below 1 spike/s:", int((unit_table.rate < 1).sum()),
      "; fewest kept trials in any of the 40 conditions:", int(unit_table.min_trials.min()))
print("Units kept:", len(keep), "of", len(unit_table))
print("Unit table columns:", data["electrodes"]["unit_columns"])
print("Electrodes:", {k: v for k, v in data["electrodes"].items() if k != "unit_columns"})

The STAR Methods report 48 neurons in the high-yield session of monkey H used for GPFA; this
session yields 48 units under the paper's selection, consistent with its being that session. The
units are single- and multi-units together; the file does not say which is which. The electrode
table has 72 electrodes in 3 groups, consistent with the Methods' 1 to 3 24-channel probes, with
no positions; the unit table has no electrode column, so no unit can be placed on a probe.

Clock check. With spikes and a trial table only, the alignment of spikes to the trial events is
checked by decoding a task variable in windows around an event, with the true labels and with the
labels of the next trial. The target direction is shown at `target_on_time`, so it can only be
decoded after it; the Set flash should evoke a transient at a fixed latency after `set_time`.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold

rng_check = np.random.default_rng(0)
win = [(-300, 0), (0, 300), (300, 600)]
y_true = (T.theta.values == 180).astype(int)
y_next = np.roll(y_true, -1)
rows = []
for a, b in win:
    X = rsg.bin_counts(SP, T.target_on_time.values, a, b, bin_ms=b - a)[:, 0, :]
    X = np.sqrt(X)
    cv = StratifiedKFold(5, shuffle=True, random_state=0)
    for lab, y in (("true", y_true), ("next trial", y_next)):
        acc = cross_val_score(LogisticRegression(C=0.1, max_iter=2000), X, y, cv=cv).mean()
        rows.append({"window after target onset (ms)": f"{a} to {b}", "labels": lab,
                     "accuracy": round(acc, 3)})
print("Decoding target direction (left/right) from spike counts, 5-fold cross-validated:")
print(pd.DataFrame(rows).pivot(index="window after target onset (ms)", columns="labels",
                               values="accuracy").to_string())
pop = rsg.bin_counts(SP, T.set_time.values, -200, 400).sum(2).mean(0) * (1000 / rsg.BIN_MS) / len(SP)
shift = rng_check.uniform(-0.3, 0.3, len(T))
pop_shift = rsg.bin_counts(SP, T.set_time.values + shift, -200, 400).sum(2).mean(0) * 50 / len(SP)
print("\nMean population rate (spikes/s per unit) in 20 ms bins from Set - 200 to Set + 400 ms:")
print("  aligned to set_time:     ", pop.round(1))
print("  times jittered +-300 ms:", pop_shift.round(1))
peak = np.argmax(pop[10:]) * 20
print(f"Peak after Set at {peak}-{peak + 20} ms: {pop[10:].max():.1f} against a pre-Set mean of "
      f"{pop[:10].mean():.1f}; jittered peak {pop_shift[10:].max():.1f}")

The target direction is decoded at 0.55 in the 300 ms before target onset and in the first 300 ms
after it, and at 0.80 from 300 to 600 ms after it; with the next trial's labels it is at 0.50 in
every window. The population rate rises from about 80 ms after Set to a peak at 180 to 200 ms, and
the peak is gone when the event times are jittered. The spikes share the clock of the trial table, as
far as these checks can tell. The analysis uses one session, so no unit is matched across sessions.

The test file holds 283 trials of the session (1.7 s segments re-timed from 0, 40 units, no ts,
tp, prior or effector), so their behaviour and labels are not in the dandiset. The paper's 17
sessions of monkey H and all of monkey G are not in it either.

## beh-slopes

> Regression slopes of tp on ts are positive and less than one in every condition, so the animals
> estimated ts and their responses were biased toward the mean of the prior.

Criterion: per prior, effector and direction (8 conditions), a weighted linear regression of tp
on ts, each trial weighted by 1 over the variance of tp at its ts; reproduced when all 8 slopes lie
between 0 and 1 with 95% intervals excluding 0 and 1. The regression has an intercept (the
criterion's "linear regression of tp on ts"). The STAR Methods write the model as
"tp = βts + ε" ("Model free analysis of behavior"), without an intercept; that reading is shown as
a variant.

In [ ]:
def weighted_slope(g, intercept=True):
    w = 1 / g.groupby("ts").tp.transform("var")
    X = sm.add_constant(g.ts.astype(float)) if intercept else g.ts.astype(float)
    r = sm.WLS(g.tp.astype(float), X, weights=w).fit()
    lo, hi = r.conf_int().loc["ts"]
    return r.params["ts"], lo, hi


rows = []
for (ds, prior), g in T.groupby(["dataset", "prior"]):
    b, lo, hi = weighted_slope(g)
    b0, lo0, hi0 = weighted_slope(g, intercept=False)
    rows.append({"dataset": ds, "prior": prior, "n": len(g), "slope": b, "ci_low": lo, "ci_high": hi,
                 "slope_no_intercept": b0, "ci0_low": lo0, "ci0_high": hi0})
slopes = pd.DataFrame(rows)
print(slopes.round(3).to_string(index=False))
inside = (slopes.slope > 0) & (slopes.slope < 1)
excl = (slopes.ci_low > 0) & (slopes.ci_high < 1)
print(f"\nWith intercept: slopes in (0, 1): {int(inside.sum())} of 8; intervals excluding 0 and 1: "
      f"{int(excl.sum())} of 8")
inside0 = (slopes.slope_no_intercept > 0) & (slopes.slope_no_intercept < 1)
excl0 = (slopes.ci0_low > 0) & (slopes.ci0_high < 1)
print(f"Variant without intercept: slopes in (0, 1): {int(inside0.sum())} of 8; intervals excluding "
      f"0 and 1: {int(excl0.sum())} of 8")

**Result: not reproduced.** With the intercept, 6 of 8 slopes lie between 0 and 1 and 5 of 8
intervals exclude 0 and 1. The Short prior slopes of both eye datasets are not below 1 (Eye Left
1.011, interval 0.896 to 1.126; Eye Right 1.110, 1.014 to 1.206). The other six slopes are 0.502 to
0.903. Without the intercept (the Methods' equation) the slopes are 0.923 to 1.072 and 2 of 8 lie
below 1, so that reading does not reproduce the claim either.

## beh-overlap-prior

> At the overlapping ts of 800 ms, biases were in opposite directions for the two priors
> (rank-sum test, p < 1e-43 in animal H over 17 sessions).

Criterion: trials with ts = 800 ms pooled over effector and direction; reproduced when the median
of tp - 800 is negative for Short and positive for Long and a two-sided rank-sum test of tp between
priors has p < 0.05. Shown per effector and direction as well.

In [ ]:
o = T[T.ts == 800]
rows = []
for ds, g in [("pooled", o)] + list(o.groupby("dataset")):
    s, l = g[g.prior == "Short"].tp, g[g.prior == "Long"].tp
    rows.append({"dataset": ds, "n_short": len(s), "n_long": len(l),
                 "median_bias_short": np.median(s - 800), "median_bias_long": np.median(l - 800),
                 "p_ranksum": stats.ranksums(s, l).pvalue})
overlap = pd.DataFrame(rows)
print(overlap.to_string(index=False, float_format=lambda v: f"{v:.3g}"))

**Result: not reproduced.** Pooled over the four datasets, the median bias at ts = 800 ms is +28.3
ms for the Short prior and +47.6 ms for the Long prior: both positive, with Long the later as the
priors predict, and the rank-sum p = 0.073. Per dataset, the Short median is negative only in Hand
Left (-61.6 against -2.8 ms, p = 0.069), and the difference is significant only in Hand Right
(29.7 against 96.8 ms, p = 0.00099).

## beh-sigmoid-increments

> Increments of tp between neighbouring ts are smaller at the extremes of the prior than near its
> mean, as the sigmoidal Bayesian estimator predicts (signed-rank test, p < 1e-11 in animal H).

Criterion: per condition the mean tp at each ts, the four increments; extreme = mean of the first
and last, middle = mean of the second and third; reproduced when middle exceeds extreme in most of
the 8 conditions and a two-sided signed-rank test over the 8 has p < 0.05.

In [ ]:
rows = []
for (ds, prior), g in T.groupby(["dataset", "prior"]):
    m = g.groupby("ts").tp.mean().reindex(rsg.PRIOR_TS[prior]).values
    inc = np.diff(m)
    rows.append({"dataset": ds, "prior": prior, "increments": inc.round(1),
                 "extreme": (inc[0] + inc[3]) / 2, "middle": (inc[1] + inc[2]) / 2})
incr = pd.DataFrame(rows)
incr["middle_minus_extreme"] = incr.middle - incr.extreme
print(incr.round(1).to_string(index=False))
n_pos = int((incr.middle_minus_extreme > 0).sum())
p_inc = signed_rank(incr.middle_minus_extreme)
print(f"\nMiddle > extreme in {n_pos} of 8; median difference "
      f"{incr.middle_minus_extreme.median():.1f} ms; signed-rank p = {p_inc:.4f}")
print("Median difference by prior (ms):", incr.groupby("prior").middle_minus_extreme.median().round(1).to_dict())

**Result: reproduced.** The middle increment exceeds the extreme one in 7 of 8 conditions (median
difference 31.3 ms; the exception is Hand Left Short, -10.7 ms), signed-rank p = 0.0156.

## beh-long-more-bias

> Biases are larger for the Long prior than for the Short prior, as scalar variability predicts;
> regression slopes are smaller for Long.

Criterion: with the slopes of beh-slopes, reproduced when the Long slope is below the Short slope
in all 4 datasets; partially reproduced for 3 of 4.

In [ ]:
sl = slopes.pivot(index="dataset", columns="prior", values="slope")
sl["long_below_short"] = sl.Long < sl.Short
sl0 = slopes.pivot(index="dataset", columns="prior", values="slope_no_intercept")
sl["Short_no_intercept"], sl["Long_no_intercept"] = sl0.Short, sl0.Long
print(sl.round(3).to_string())
print(f"\nLong slope below Short: {int(sl.long_below_short.sum())} of 4 "
      f"(variant without intercept: {int((sl0.Long < sl0.Short).sum())} of 4)")

**Result: reproduced.** The Long slope is below the Short slope in all 4 datasets (0.845 against
1.011, 0.903 against 1.110, 0.502 against 0.612, 0.612 against 0.861). Without the intercept it is
below in 3 of 4.

## beh-bayes-model

> A Bayesian observer model with scalar measurement and production noise fit to the behaviour
> captures the bias and the variance of tp in all conditions.

Criterion: the model fit per dataset (wm, wp, bShort, bLong). Across the 10 prior and ts conditions,
the R2 of the observed mean bias by the model's predicted bias, and the Pearson correlation of the
observed and predicted SD of tp. Reproduced when the bias R2 is at least 0.8 and the SD correlation
positive in all 4 datasets; partially when only one of the two holds in all 4.

"R2 of the observed by the predicted" can be read as (a) 1 - SS_residual / SS_total with the
prediction taken as it is, or (b) the squared correlation of observed and predicted (a regression of
one on the other). Both are shown; they are judged together.

In [ ]:
fits, preds = {}, []
for ds, g in T.groupby("dataset"):
    fits[ds] = rsg.fit_bls(g)
    for prior in PRIORS:
        p = rsg.bls_predict(fits[ds], prior, rsg.PRIOR_TS[prior])
        obs = g[g.prior == prior].groupby("ts").tp.agg(["mean", "std", "size"])
        p["dataset"], p["prior"] = ds, prior
        p["obs_mean"], p["obs_sd"], p["n"] = obs["mean"].values, obs["std"].values, obs["size"].values
        preds.append(p)
preds = pd.concat(preds, ignore_index=True)
preds["obs_bias"], preds["pred_bias"] = preds.obs_mean - preds.ts, preds.tp_mean - preds.ts
print(pd.DataFrame(fits).T.round(3).to_string())
rows = []
for ds, p in preds.groupby("dataset"):
    ss_res = np.sum((p.obs_bias - p.pred_bias) ** 2)
    ss_tot = np.sum((p.obs_bias - p.obs_bias.mean()) ** 2)
    rows.append({"dataset": ds, "bias_R2_a": 1 - ss_res / ss_tot,
                 "bias_R2_b": stats.pearsonr(p.obs_bias, p.pred_bias)[0] ** 2,
                 "sd_r": stats.pearsonr(p.obs_sd, p.tp_sd)[0],
                 "sd_p": stats.pearsonr(p.obs_sd, p.tp_sd)[1]})
bayes = pd.DataFrame(rows)
print()
print(bayes.round(3).to_string(index=False))
print(preds[["dataset", "prior", "ts", "n", "obs_bias", "pred_bias", "obs_sd", "tp_sd"]].round(1).to_string(index=False))
print(f"\nBias R2 >= 0.8: reading (a) {int((bayes.bias_R2_a >= 0.8).sum())} of 4, reading (b) "
      f"{int((bayes.bias_R2_b >= 0.8).sum())} of 4; SD correlation positive: {int((bayes.sd_r > 0).sum())} of 4")

**Result: partially reproduced.** The SD correlation is positive in all 4 datasets (0.277 to 0.909),
but the bias R2 reaches 0.8 in 2 of 4 under both readings (Hand Left 0.978, Hand Right 0.865; Eye
Left 0.670, Eye Right 0.122). The fitted wm is 0.013 to 0.116 and wp 0.090 to 0.106; the eye
datasets have small wm and small, irregular biases.

## glm-prior-ts

> About 30% of neurons were modulated by elapsed time (27% in monkey H) and more than 60% were
> sensitive to the prior (65% in monkey H), by a Poisson GLM of spike counts before Set compared by
> BIC.

Criterion: the selected units; counts in the 80 ms before each of the Short prior's ts values (480
to 800 ms after Ready) from every trial whose Set comes after that time, both priors. Full model:
log rate = prior + 5 time indicators + effector + direction; prior-dependent when its BIC is below
that of the model without the prior. For ts: a model per prior with that prior's 5 time indicators
plus effector and direction, against the model without the time indicators (intercept, effector
and direction, the STAR Methods' Equation 4); ts-dependent when the BIC favours the full model in
either prior. Expected 65 +- 6.5% (prior) and 27 +- 2.7% (ts).

A trial whose ts equals the window's end is included: its window is the 80 ms "immediately before
Set" of the STAR Methods ("Generalized linear model"), and the attrition rule there counts "all
trials in which Set occurred after t". Windows are placed at the nominal times after `ready_time`.

In [ ]:
from statsmodels.stats.proportion import proportion_confint


def glm_rows(t, times):
    """Rows (trial, window end) with ts >= window end, and counts for every unit."""
    rows, counts = [], []
    for T_end in times:
        g = t[t.ts >= T_end]
        c = rsg.bin_counts(SP, g.ready_time.values, T_end - 80, T_end, bin_ms=80)[:, 0, :]
        counts.append(c)
        rows.append(pd.DataFrame({"time": T_end, "long": (g.prior == "Long").astype(float).values,
                                  "hand": (~g.is_eye.astype(bool)).astype(float).values,
                                  "left": (g.theta == 180).astype(float).values}))
    return pd.concat(rows, ignore_index=True), np.vstack(counts)


def bic(y, X):
    r = sm.GLM(y, X, family=sm.families.Poisson()).fit()
    if not r.converged:
        raise RuntimeError("GLM did not converge")
    return -2 * r.llf + X.shape[1] * np.log(len(y))


def design(rows, prior=True, time=True):
    cols = []
    if time:
        cols.append(pd.get_dummies(rows.time, prefix="t").astype(float))
    else:
        cols.append(pd.DataFrame({"const": np.ones(len(rows))}))
    if prior:
        cols.append(rows[["long"]])
    cols.append(rows[["hand", "left"]])
    return pd.concat(cols, axis=1).values


rows_p, y_p = glm_rows(T, rsg.SHORT_TS)
per_prior = {p: glm_rows(T[T.prior == p], rsg.PRIOR_TS[p]) for p in PRIORS}
print("Rows in the prior model:", len(rows_p), "; by window end:", rows_p.time.value_counts().sort_index().to_dict())
res = []
for j in range(len(SP)):
    r = {"unit": int(unit_table[unit_table.kept].unit.values[j])}
    r["prior_dep"] = bic(y_p[:, j], design(rows_p)) < bic(y_p[:, j], design(rows_p, prior=False))
    for p in PRIORS:
        rr, yy = per_prior[p]
        r[f"ts_dep_{p}"] = bic(yy[:, j], design(rr, prior=False)) < bic(yy[:, j], design(rr, prior=False, time=False))
    res.append(r)
glm = pd.DataFrame(res)
glm["ts_dep"] = glm.ts_dep_Short | glm.ts_dep_Long
n = len(glm)
for col, label, expv in (("prior_dep", "prior", 65), ("ts_dep", "ts (either prior)", 27),
                         ("ts_dep_Short", "ts (Short model)", None), ("ts_dep_Long", "ts (Long model)", None)):
    k = int(glm[col].sum())
    lo, hi = proportion_confint(k, n, method="beta")
    print(f"{label:>18}: {k} of {n} = {100 * k / n:.1f}% (95% CI {100 * lo:.1f} to {100 * hi:.1f})"
          + (f"; expected {expv} +- {expv / 10:.1f}" if expv else ""))
print("Both prior- and ts-dependent:", int((glm.prior_dep & glm.ts_dep).sum()),
      "; neither:", int((~glm.prior_dep & ~glm.ts_dep).sum()))
held = data["units"].loc[glm.unit, "heldout"].values
print("Among the 40 benchmark held-in units kept vs held-out units kept:",
      f"prior {glm.prior_dep[~held].mean():.2f} / {glm.prior_dep[held].mean():.2f},",
      f"ts {glm.ts_dep[~held].mean():.2f} / {glm.ts_dep[held].mean():.2f}")

**Result: not reproduced.** 38 of 48 units (79.2%, 95% interval 65.0 to 89.5) are prior-dependent,
above the 65 +- 6.5% range, and 22 of 48 (45.8%, 31.4 to 60.8) are ts-dependent, above 27 +- 2.7%.
Both fractions are higher than the paper's, which pool 536 units of 17 sessions.

## dfr-overlap-support

> At the overlapping ts of 800 ms, firing rates differ between priors, with the largest difference
> during the support of the prior, between 480 and 800 ms after Ready.

Criterion: per dataset, ts = 800 trials; per unit the difference of mean rates between priors in
160 ms bins from 320 ms before Ready to 640 ms after Set; the root mean square over units, averaged
over the 4 datasets. Reproduced when the largest value falls in a bin within 480 to 800 ms after
Ready and exceeds the 95th percentile of the largest value over 200 shuffles of the prior labels
among ts = 800 trials. Bins up to Set are aligned to `ready_time`, those after Set to `set_time`
(ts = 800 ms, so the bins are contiguous in nominal time).

In [ ]:
rng_dfr = np.random.default_rng(7)
o = T[T.ts == 800]
labels = ([f"R{a}..{a + 160}" for a in range(-320, 800, 160)]
          + [f"S{a}..{a + 160}" for a in range(0, 640, 160)])
counts_dfr = {}
for ds, g in o.groupby("dataset"):
    a = rsg.bin_counts(SP, g.ready_time.values, -320, 800, bin_ms=160)
    b = rsg.bin_counts(SP, g.set_time.values, 0, 640, bin_ms=160)
    counts_dfr[ds] = (np.concatenate([a, b], axis=1) / 0.16, (g.prior == "Long").values)


def rms_curve(shuffle_rng=None):
    curves = []
    for ds, (rates, is_long) in counts_dfr.items():
        lab = shuffle_rng.permutation(is_long) if shuffle_rng is not None else is_long
        diff = rates[lab].mean(0) - rates[~lab].mean(0)
        curves.append(np.sqrt((diff ** 2).mean(1)))
    return np.array(curves)


curves = rms_curve()
mean_curve = curves.mean(0)
null_max = np.array([rms_curve(rng_dfr).mean(0).max() for _ in range(200)])
print(pd.DataFrame(curves, index=list(counts_dfr), columns=labels).round(2).to_string())
print("mean over datasets:", dict(zip(labels, mean_curve.round(2))))
k = int(np.argmax(mean_curve))
print(f"\nLargest mean RMS difference: {mean_curve[k]:.2f} spikes/s in bin {labels[k]} "
      f"(support of the prior = bins R480..640 and R640..800)")
print(f"Shuffle null of the largest value: median {np.median(null_max):.2f}, 95th percentile "
      f"{np.percentile(null_max, 95):.2f}")
print("Per-dataset bin of the largest value:", {ds: labels[i] for ds, i in zip(counts_dfr, curves.argmax(1))})
print("Trials per dataset (Short, Long):", {ds: (int((~l).sum()), int(l.sum())) for ds, (_, l) in counts_dfr.items()})

**Result: reproduced.** The mean RMS difference between priors peaks at 6.86 spikes/s in the bin
480 to 640 ms after Ready, above the 95th percentile of the shuffled maxima (3.09). By dataset the
peak is in the support for three datasets and in the first 160 ms after Set for Eye Left.

## Population trajectories

Shared by the claims below. Per dataset, trial-averaged rates (20 ms bins, Gaussian SD 40 ms) of
the 48 units. Estimation epoch: from Ready to the prior's longest ts, with attrition. Production
epoch: from Set to the shortest Set-to-Go time of each ts. PCA per dataset and epoch on both priors
concatenated, keeping the components that reach 75% of the variance (STAR Methods, "Principal
component analysis").

In [ ]:
EST, EST_N, PCA_E, K_E, PROD, PCA_P, K_P = {}, {}, {}, {}, {}, {}, {}
for ds, g in T.groupby("dataset"):
    EST[ds], EST_N[ds], PROD[ds] = {}, {}, {}
    for prior in PRIORS:
        gp = g[g.prior == prior]
        EST[ds][prior], EST_N[ds][prior] = rsg.estimation_rates(SP, gp, max(rsg.PRIOR_TS[prior]))
        for ts in rsg.PRIOR_TS[prior]:
            PROD[ds][(prior, ts)] = rsg.production_rates(SP, gp[gp.ts == ts])
    PCA_E[ds], K_E[ds] = rsg.fit_pca([EST[ds][p] for p in PRIORS])
    PCA_P[ds], K_P[ds] = rsg.fit_pca(list(PROD[ds].values()))
print("Production trajectory lengths (bins of 20 ms) per dataset:")
print(pd.DataFrame({ds: {f"{p} {ts}": v.shape[0] for (p, ts), v in PROD[ds].items()} for ds in DS}).to_string())
print("Rate ranges (spikes/s), estimation:", {ds: (round(min(v.min() for v in EST[ds].values()), 1),
      round(max(v.max() for v in EST[ds].values()), 1)) for ds in DS})


def est_states(ds, prior):
    """Estimation-epoch states in the dataset's estimation PC space."""
    return PCA_E[ds].transform(EST[ds][prior])[:, :K_E[ds]]


def prod_states(ds, key, rates=None):
    r = PROD[ds][key] if rates is None else rates
    return PCA_P[ds].transform(r)[:, :K_P[ds]]

## pca-dimensionality

> Population activity is low dimensional; 3 to 4 principal components in the estimation epoch and
> 5 to 10 in the production epoch explain about 75% of the variance.

Criterion: the number of components to reach 75%, per dataset and epoch; ranges widened by 10%:
3 or 4 (estimation) and 5 to 11 (production). Reproduced when all 8 values fall in range; partially
when at least 4 do.

In [ ]:
rows = []
for ds in DS:
    for ep, pca, k, rng_ok in (("estimation", PCA_E[ds], K_E[ds], (3, 4)), ("production", PCA_P[ds], K_P[ds], (5, 11))):
        cum = np.cumsum(pca.explained_variance_ratio_)
        rows.append({"dataset": ds, "epoch": ep, "n_PC_75": k, "in_range": rng_ok[0] <= k <= rng_ok[1],
                     "cum_var_first_6": cum[:6].round(3)})
dims = pd.DataFrame(rows)
print(dims.to_string(index=False))
print(f"\nIn range: {int(dims.in_range.sum())} of 8")
print("Without smoothing (variant):", {ds: (rsg.fit_pca([rsg.estimation_rates(SP, T[(T.dataset == ds) & (T.prior == p)], max(rsg.PRIOR_TS[p]), smoothed=False)[0] for p in PRIORS])[1])
      for ds in DS}, "(estimation)")
soft = {}
for ds in DS:
    g = T[T.dataset == ds]
    blocks = [rsg.estimation_rates(SP, g[g.prior == p], max(rsg.PRIOR_TS[p]))[0] for p in PRIORS]
    rng_u = np.vstack(blocks).max(0) - np.vstack(blocks).min(0)
    soft[ds] = rsg.fit_pca([b / (rng_u + 5) for b in blocks])[1]
print("Soft-normalised rates, rate / (range + 5) (variant, not the claims file's method):", soft, "(estimation)")

**Result: not reproduced.** 1 of 8 values is in range: 2 components reach 75% in the estimation epoch
of every dataset, and 2, 2, 5 and 4 in the production epoch (Eye Right, Eye Left, Hand Right, Hand
Left). The first component alone explains 0.51 to 0.64 of the variance in the estimation epoch.
Without smoothing the estimation epoch needs 3, 3, 4 and 6 components, and with soft-normalised
rates 3 in each; neither variant is the claims file's method.

## encoding-r2

> Projections of estimation-epoch neural states onto the encoding axis match the Bayesian
> estimates from the model fitted to the behaviour (R2 = 0.993 for the Short prior, 0.996 for the
> Long prior, monkey H, Hand Left).

Criterion: weighted regression Xu = a + b te over the 20 ms time points of the prior's support,
each point weighted by the number of trials it averages; te(t) is the model's mean estimate at ts =
t, from the fits of beh-bayes-model. Readings: (a) the Hand Left dataset, (b) the mean over the 4
datasets. Reproduced when under one reading both R2 are within tolerance (at least 0.894 and
0.896); partially when one is. Reference: R2 for 1000 random unit vectors in the same PC space.

In [ ]:
rng_enc = np.random.default_rng(11)
ENC = {}
rows = []
for ds in DS:
    for prior in PRIORS:
        lo, hi = min(rsg.PRIOR_TS[prior]), max(rsg.PRIOR_TS[prior])
        X = est_states(ds, prior)
        u = rsg.encoding_axis(X, lo, hi)
        tt = np.arange(lo, hi + 1, rsg.BIN_MS)
        idx = tt // rsg.BIN_MS - 1
        xu, w = X[idx] @ u, EST_N[ds][prior][idx]
        te = rsg.bls_predict(fits[ds], prior, tt).te.values
        (_, b_te), r2_te, fit_te = rsg.wls(xu, te, w)
        (_, b_ts), r2_ts, fit_ts = rsg.wls(xu, tt, w)
        rand = rng_enc.normal(size=(1000, X.shape[1]))
        rand /= np.linalg.norm(rand, axis=1, keepdims=True)
        r2_rand = np.array([rsg.wls(X[idx] @ v, te, w)[1] for v in rand])
        ENC[(ds, prior)] = dict(t=tt, xu=xu, w=w, te=te, X=X, u=u, slope_ts=b_ts,
                                rmse_bayes=np.sqrt(np.mean((xu - fit_te) ** 2)),
                                rmse_lin=np.sqrt(np.mean((xu - fit_ts) ** 2)),
                                wrmse_bayes=np.sqrt(np.sum(w * (xu - fit_te) ** 2) / w.sum()),
                                wrmse_lin=np.sqrt(np.sum(w * (xu - fit_ts) ** 2) / w.sum()))
        rows.append({"dataset": ds, "prior": prior, "n_PC": X.shape[1], "points": len(tt),
                     "R2_te": r2_te, "R2_ts": r2_ts, "R2_random_median": np.median(r2_rand),
                     "R2_random_95pct": np.percentile(r2_rand, 95),
                     "frac_random_above": np.mean(r2_rand >= r2_te)})
enc = pd.DataFrame(rows)
print(enc.round(3).to_string(index=False))
hl = enc[enc.dataset == "Hand Left"].set_index("prior").R2_te
mn = enc.groupby("prior").R2_te.mean()
print(f"\nReading (a) Hand Left: Short {hl.Short:.3f}, Long {hl.Long:.3f}")
print(f"Reading (b) mean over datasets: Short {mn.Short:.3f}, Long {mn.Long:.3f}")
print("Within tolerance (>= 0.894 Short, >= 0.896 Long): (a)", [bool(hl.Short >= 0.894), bool(hl.Long >= 0.896)],
      "; (b)", [bool(mn.Short >= 0.894), bool(mn.Long >= 0.896)])
fig, axes = plt.subplots(1, 4, figsize=(14, 3))
for ax, ds in zip(axes, DS):
    for prior, col in (("Short", "tab:red"), ("Long", "tab:blue")):
        e = ENC[(ds, prior)]
        ax.plot(e["t"], e["xu"], "o", color=col, ms=3)
        ax.plot(e["t"], rsg.wls(e["xu"], e["te"], e["w"])[2], "-", color=col)
    ax.set_title(ds); ax.set_xlabel("t after Ready (ms)")
axes[0].set_ylabel("Xu (spikes/s)")
plt.tight_layout(); plt.show()
print("Plotted Xu ranges:", {f"{d} {p}": (round(e['xu'].min(), 1), round(e['xu'].max(), 1)) for (d, p), e in ENC.items()})

**Result: reproduced.** Under reading (a), Hand Left, R2 is 0.994 (Short) and 0.986 (Long); under
reading (b), the mean over datasets, 0.955 and 0.986. All are within tolerance. The estimation PC
space has 2 dimensions in every dataset, and random unit vectors in it also give high R2 (medians
0.749 to 0.987); the fraction of random vectors at or above the observed R2 is 0.004 to 0.434, and
R2 with ts in place of te is within 0.013 of R2 with te in every case.

## encoding-bayes-beats-linear

> Neural projections are better explained by the Bayesian model than by a linear function of ts
> with the same number of parameters, for both priors (signed-rank test on RMSE, p = 0.008 for
> each prior).

Criterion: RMSE of the weighted fits Xu = a + b te and Xu = a + b ts, per dataset and prior;
reproduced when the Bayesian RMSE is lower in all 4 datasets for both priors; partially when it is
for one prior. The RMSE is the root mean square of the residuals ("Root Mean Squared Error between
the actual and predicted Xu", STAR Methods); the weighted RMSE is shown as a variant.

In [ ]:
rows = [{"dataset": ds, "prior": p, "rmse_bayes": e["rmse_bayes"], "rmse_linear": e["rmse_lin"],
         "delta": e["rmse_bayes"] - e["rmse_lin"], "weighted_delta": e["wrmse_bayes"] - e["wrmse_lin"]}
        for (ds, p), e in ENC.items()]
rm = pd.DataFrame(rows)
print(rm.round(4).to_string(index=False))
for p in PRIORS:
    r = rm[rm.prior == p]
    print(f"{p}: Bayesian lower in {int((r.delta < 0).sum())} of 4 (weighted RMSE: "
          f"{int((r.weighted_delta < 0).sum())} of 4); median delta {r.delta.median():.4f}; "
          f"signed-rank p = {signed_rank(r.delta):.3f}")

**Result: not reproduced.** The Bayesian RMSE is lower in 3 of 4 datasets for each prior, not in all
4: it is higher for Hand Right Short (+0.180) and Hand Left Long (+0.081). The weighted RMSE gives the
same counts.

## short-in-long-control

> In the Long prior during the period matched to the support of the Short prior ("Short In Long"),
> there is no sigmoidal representation; the Bayesian model is not better than the linear one
> (signed-rank test, p = 0.74).

Criterion (absence, judged on the paper's test): Long prior states from 480 to 800 ms after Ready,
an axis from the state at 480 to that at 800 ms, compared with the Short prior's te and with ts as
above; reproduced when a two-sided signed-rank test of the RMSE difference over the 4 datasets has
p >= 0.05. The axis is built as u is, from the averaged difference vectors (the STAR Methods
"applied the same projection analysis"); the single vector from 480 to 800 ms is shown as a
variant. All Long trials have Set at or after 800 ms, so every point has the same weight.

In [ ]:
rows = []
for ds in DS:
    X = est_states(ds, "Long")
    tt = np.arange(480, 801, rsg.BIN_MS)
    idx = tt // rsg.BIN_MS - 1
    w = EST_N[ds]["Long"][idx]
    te = rsg.bls_predict(fits[ds], "Short", tt).te.values
    for axis_name, u in (("averaged", rsg.encoding_axis(X, 480, 800)),
                         ("single vector", (X[idx[-1]] - X[idx[0]]) / np.linalg.norm(X[idx[-1]] - X[idx[0]]))):
        xu = X[idx] @ u
        rb = np.sqrt(np.mean((xu - rsg.wls(xu, te, w)[2]) ** 2))
        rl = np.sqrt(np.mean((xu - rsg.wls(xu, tt, w)[2]) ** 2))
        rows.append({"dataset": ds, "axis": axis_name, "rmse_bayes": rb, "rmse_linear": rl, "delta": rb - rl,
                     "R2_te": rsg.wls(xu, te, w)[1], "R2_ts": rsg.wls(xu, tt, w)[1]})
sil = pd.DataFrame(rows)
print(sil.round(4).to_string(index=False))
for ax_name, r in sil.groupby("axis", sort=False):
    print(f"{ax_name}: Bayesian lower in {int((r.delta < 0).sum())} of 4; median delta {r.delta.median():.4f};"
          f" signed-rank p = {signed_rank(r.delta):.3f}")
short_delta = rm[rm.prior == "Short"].set_index("dataset").delta
print("Short prior deltas for comparison:", short_delta.round(4).to_dict())
print("Short In Long minus Short delta, per dataset:",
      (sil[sil.axis == "averaged"].set_index("dataset").delta - short_delta).round(4).to_dict())

**Result: reproduced, by the paper's test.** The signed-rank p is 0.250 for the RMSE difference
over 4 datasets (and with the single-vector axis), so the test does not find a difference; with 4
datasets it cannot reach p < 0.05. The effect it would test is not absent: the Bayesian RMSE is lower
in 3 of 4 datasets for Short In Long (median difference -0.109), as for the Short prior itself (3 of
4, median -0.289).

## neural-long-more-bias

> Neural projections show more bias for the Long prior than for the Short prior, as a smaller
> regression slope of Xu on ts (signed-rank test, p = 0.008).

Criterion: the slope of the weighted regression of Xu (spikes/s, shared PC space of the dataset) on
ts (ms), per dataset and prior; reproduced when Long is below Short in all 4 datasets; partially
for 3.

In [ ]:
nb_sl = pd.DataFrame({p: {ds: ENC[(ds, p)]["slope_ts"] for ds in DS} for p in PRIORS})
nb_sl["long_below_short"] = nb_sl.Long < nb_sl.Short
print(nb_sl.round(4).to_string())
print(f"\nLong below Short in {int(nb_sl.long_below_short.sum())} of 4; "
      f"signed-rank p (Short - Long) = {signed_rank(nb_sl.Short - nb_sl.Long):.3f}")

**Result: reproduced.** The Long slope is below the Short slope in all 4 datasets; in Hand Right
they are nearly equal (0.0496 against 0.0504 spikes/s per ms). Signed-rank p = 0.125.

## speed-vs-projection

> Larger projections on the encoding axis at Set are followed by slower neural trajectories in the
> production epoch (Pearson correlation, rho Short = -0.74, rho Long = -0.51).

Criterion: per prior, Xu at t = ts against the speed of the production trajectory of that ts, pooled
over the 4 datasets (20 points); speed is the mean Euclidean distance between successive 20 ms
states in the production PC space, from Set to the shortest tp of that ts (claims header).
Reproduced when both correlations are within tolerance (-0.74 +- 0.074, -0.51 +- 0.051); partially
when both are negative with p < 0.05 but one is outside tolerance, or when only one is within
tolerance. Variants: correlations within each dataset; speed divided by the mean Set-to-Go time of
the ts (the STAR Methods' "divided by the duration separating Set and the time of Go").

In [ ]:
rows = []
for ds in DS:
    g = T[T.dataset == ds]
    for (prior, ts) in PROD[ds]:
        Y = prod_states(ds, (prior, ts))
        e = ENC[(ds, prior)]
        gg = g[(g.prior == prior) & (g.ts == ts)]
        rows.append({"dataset": ds, "prior": prior, "ts": ts,
                     "xu_at_set": e["xu"][list(e["t"]).index(ts)],
                     "speed": np.mean(np.linalg.norm(np.diff(Y, axis=0), axis=1)),
                     "set_to_go_s": np.mean(gg.go_time - gg.set_time), "tp_mean": gg.tp.mean()})
SPD = pd.DataFrame(rows)
SPD["speed_per_s"] = SPD.speed / SPD.set_to_go_s
print(SPD.round(3).to_string(index=False))


def corr_table(x, y):
    out = []
    for prior in PRIORS:
        s = SPD[SPD.prior == prior]
        r, p = stats.pearsonr(s[x], s[y])
        per = {ds: round(stats.pearsonr(q[x], q[y])[0], 2) for ds, q in s.groupby("dataset")}
        out.append({"prior": prior, "n": len(s), "r_pooled": round(r, 3), "p": p, "r_within_dataset": per})
    return pd.DataFrame(out)


print("\nSpeed (mean step) against Xu at Set:")
print(corr_table("xu_at_set", "speed").to_string(index=False))
print("\nVariant, speed divided by the Set-to-Go time:")
print(corr_table("xu_at_set", "speed_per_s").to_string(index=False))

**Result: partially reproduced.** Both pooled correlations are negative and significant, but
stronger than the paper's and outside tolerance: r = -0.926 (Short, p = 4.8e-09) and -0.834 (Long,
p = 4.9e-06). Within datasets r is -0.74 to -1.0. Speed divided by the Set-to-Go time gives -0.916
and -0.908.

## initial-speed-biased

> The initial speed of production trajectories (Set to Set + 400 ms) is biased toward the middle of
> the prior; its dependence on ts is shallower than the unbiased profile proportional to 1/ts
> (signed-rank test, p < 1e-3 over conditions and animals).

Criterion: per dataset and prior, the mean step distance between Set and Set + 400 ms (trials with
tp of at least 400 ms), the slope of its regression on ts, and the slope of the unbiased profile
speed(middle ts) x middle ts / ts. Reproduced when the measured slope is shallower (less negative)
than the unbiased one in most of the 8 and a two-sided signed-rank test of the difference has
p < 0.05. States are in the dataset's production PC space.

In [ ]:
rows = []
for ds in DS:
    g = T[(T.dataset == ds) & (T.tp >= 400)]
    for prior in PRIORS:
        tsv = np.array(rsg.PRIOR_TS[prior], float)
        sp = []
        for ts in rsg.PRIOR_TS[prior]:
            Y = prod_states(ds, None, rsg.production_rates(SP, g[(g.prior == prior) & (g.ts == ts)], length_ms=400))
            sp.append(np.mean(np.linalg.norm(np.diff(Y, axis=0), axis=1)))
        sp = np.array(sp)
        unb = sp[2] * tsv[2] / tsv
        rows.append({"dataset": ds, "prior": prior, "speeds": sp.round(2),
                     "slope_measured": np.polyfit(tsv, sp, 1)[0], "slope_unbiased": np.polyfit(tsv, unb, 1)[0]})
ini = pd.DataFrame(rows)
ini["difference"] = ini.slope_measured - ini.slope_unbiased
print(ini.to_string(index=False, formatters={c: "{:.5f}".format for c in ["slope_measured", "slope_unbiased", "difference"]}))
print(f"\nMeasured shallower than unbiased (difference > 0): {int((ini.difference > 0).sum())} of 8; "
      f"measured slope negative: {int((ini.slope_measured < 0).sum())} of 8; "
      f"signed-rank p = {signed_rank(ini.difference):.4f}")

**Result: reproduced.** The measured slope of initial speed on ts is shallower than the unbiased
one in 7 of 8 dataset-priors (the exception is Eye Right Long), signed-rank p = 0.039. All 8
measured slopes are negative.

## speed-predicts-tp

> The speed of neural trajectories in the production epoch predicts the mean tp (Pearson
> correlation, rho Short = -0.58, rho Long = -0.52).

Criterion: per prior, the mean tp of each dataset and ts against the speed of that ts (as in
speed-vs-projection), pooled over the 4 datasets (20 points); same rule as speed-vs-projection.
Variant: the trial-by-trial correlation within each prior, effector and direction. Here each trial's
rates (20 ms bins, Gaussian SD 40 ms) from Set to its Go are projected into the dataset's production
PC space; speed and tp are z-scored within each ts (as the STAR Methods do for Figure S6) and
correlated. This single-trial speed is a stand-in for the benchmark's model-inferred rates.

In [ ]:
print("Mean tp against speed (mean step):")
print(corr_table("speed", "tp_mean").to_string(index=False))
print("\nVariant, speed divided by the Set-to-Go time:")
print(corr_table("speed_per_s", "tp_mean").to_string(index=False))
rows = []
for ds in DS:
    g = T[T.dataset == ds]
    for prior in PRIORS:
        zs, zt = [], []
        for ts in rsg.PRIOR_TS[prior]:
            gg = g[(g.prior == prior) & (g.ts == ts)]
            s_tr = []
            for r in gg.itertuples():
                nb = int((r.go_time - r.set_time) * 1000 // rsg.BIN_MS)
                c = rsg.bin_counts(SP, np.array([r.set_time]), 0, nb * rsg.BIN_MS)[0] * (1000 / rsg.BIN_MS)
                Y = PCA_P[ds].transform(rsg.smooth(c))[:, :K_P[ds]]
                s_tr.append(np.mean(np.linalg.norm(np.diff(Y, axis=0), axis=1)))
            zs.append(stats.zscore(s_tr)); zt.append(stats.zscore(gg.tp.values))
        r, p = stats.pearsonr(np.concatenate(zs), np.concatenate(zt))
        rows.append({"dataset": ds, "prior": prior, "trials": len(np.concatenate(zs)), "r_trial": r, "p": p})
trial_corr = pd.DataFrame(rows)
print("\nVariant, trial by trial (z-scored within ts):")
print(trial_corr.round(4).to_string(index=False))
print(f"Negative in {int((trial_corr.r_trial < 0).sum())} of 8; mean r {trial_corr.r_trial.mean():.3f}")

**Result: partially reproduced.** Speed and mean tp are negatively correlated, r = -0.720 (Short,
p = 0.00034) and -0.699 (Long, p = 0.00060), both outside tolerance (stronger than -0.58 and -0.52).
Within datasets r is -0.83 to -0.99. Trial by trial, z-scored within ts, r is negative in 8 of 8
conditions (mean -0.268).

## alt-speed-unmodulated

> Instantaneous speed of neural trajectories in the full neural space is not modulated during the
> support of each prior (signed-rank test for zero regression slopes, p = 0.64 for Short, p = 0.08
> for Long).

Criterion (absence, judged on the paper's test): per dataset and prior, the distance between
successive 20 ms states of the smoothed rates of all selected units over the support of the prior
(with attrition), regressed on time; reproduced when a two-sided signed-rank test of the 4 slopes
against zero has p >= 0.05 for each prior. With 4 datasets the test cannot reach p < 0.05; the
slopes and their bootstrap intervals over trials (200 resamples of trials within dataset and
prior) are the effect size.

In [ ]:
rng_spd = np.random.default_rng(23)


def support_speed_slope(ds, prior, t):
    lo, hi = min(rsg.PRIOR_TS[prior]), max(rsg.PRIOR_TS[prior])
    R, _ = rsg.estimation_rates(SP, t, hi)
    idx = np.arange(lo, hi + 1, rsg.BIN_MS) // rsg.BIN_MS - 1
    steps = np.linalg.norm(np.diff(R[idx], axis=0), axis=1)
    times = np.arange(lo + rsg.BIN_MS, hi + 1, rsg.BIN_MS)
    return np.polyfit(times, steps, 1)[0], steps


rows = []
for ds in DS:
    for prior in PRIORS:
        g = T[(T.dataset == ds) & (T.prior == prior)]
        slope, steps = support_speed_slope(ds, prior, g)
        boot = [support_speed_slope(ds, prior, g.iloc[rng_spd.integers(0, len(g), len(g))])[0] for _ in range(200)]
        rows.append({"dataset": ds, "prior": prior, "mean_step": steps.mean(), "slope_per_s": slope * 1000,
                     "boot_lo": np.percentile(boot, 2.5) * 1000, "boot_hi": np.percentile(boot, 97.5) * 1000,
                     "relative_change_over_support": slope * (len(steps) - 1) * rsg.BIN_MS / steps.mean()})
asp = pd.DataFrame(rows)
print(asp.round(3).to_string(index=False))
for prior in PRIORS:
    s = asp[asp.prior == prior]
    print(f"{prior}: slopes > 0 in {int((s.slope_per_s > 0).sum())} of 4; bootstrap intervals excluding 0: "
          f"{int(((s.boot_lo > 0) | (s.boot_hi < 0)).sum())} of 4; signed-rank p = {signed_rank(s.slope_per_s):.3f}")

# Variant: the same trials at every point of the support (only trials with the prior's longest ts),
# so that the number of trials averaged, and the noise it leaves in the rates, does not change.
rows = []
for ds in DS:
    for prior in PRIORS:
        g = T[(T.dataset == ds) & (T.prior == prior) & (T.ts == max(rsg.PRIOR_TS[prior]))]
        slope, steps = support_speed_slope(ds, prior, g)
        rows.append({"dataset": ds, "prior": prior, "trials": len(g), "mean_step": steps.mean(),
                     "slope_per_s": slope * 1000})
asp_fixed = pd.DataFrame(rows)
print("\nVariant without attrition (trials of the longest ts only):")
print(asp_fixed.round(3).to_string(index=False))
for prior in PRIORS:
    s = asp_fixed[asp_fixed.prior == prior]
    print(f"{prior}: slopes > 0 in {int((s.slope_per_s > 0).sum())} of 4; signed-rank p = {signed_rank(s.slope_per_s):.3f}")
print("Trials averaged at the start and end of the support (main analysis):",
      {f"{ds} {p}": (int(EST_N[ds][p][min(rsg.PRIOR_TS[p]) // 20 - 1]), int(EST_N[ds][p][max(rsg.PRIOR_TS[p]) // 20 - 1]))
       for ds in DS for p in PRIORS})

**Result: reproduced, by the paper's test.** The signed-rank p is 0.125 for each prior, which with 4
datasets is the smallest the test can give. The effect the claim denies is present: speed rises
through the support in all 8 dataset-priors (slopes 3.95 to 10.61 per s, bootstrap intervals
excluding 0 in 8 of 8; 30% to 82% over the support). Part of the rise comes from attrition, which
leaves fewer trials, and noisier rates, late in the support: with the trials of the longest ts only,
the slopes are 0.97 to 5.29 per s in 7 of 8 and -1.07 in Hand Right Long.

## alt-transient-parallel

> In the first 200 ms after Set, trajectories for different ts run parallel; their distance to the
> middle ts trajectory does not change (signed-rank test for zero slopes, p = 0.92 for the shortest
> ts, p = 0.13 for the longest).

Criterion (absence, judged on the paper's test): PCA per dataset on Set to Set + 200 ms, both
priors, at least 75% of variance; per dataset and prior the distance from the shortest and the
longest ts trajectory to the middle ts trajectory at each 20 ms bin, regressed on time; reproduced
when a two-sided signed-rank test of the 8 slopes against zero has p >= 0.05 for both.

In [ ]:
rows = []
for ds in DS:
    g = T[T.dataset == ds]
    tr = {(p, ts): rsg.production_rates(SP, g[(g.prior == p) & (g.ts == ts)], length_ms=200)
          for p in PRIORS for ts in rsg.PRIOR_TS[p]}
    pca, k = rsg.fit_pca(list(tr.values()))
    Z = {key: pca.transform(v)[:, :k] for key, v in tr.items()}
    times = np.arange(1, 11) * rsg.BIN_MS
    for prior in PRIORS:
        tsv = rsg.PRIOR_TS[prior]
        mid = Z[(prior, tsv[2])]
        for which, ts in (("shortest", tsv[0]), ("longest", tsv[-1])):
            dist = np.linalg.norm(Z[(prior, ts)] - mid, axis=1)
            rows.append({"dataset": ds, "prior": prior, "ts": which, "n_PC": k,
                         "dist_first": dist[0], "dist_last": dist[-1], "slope_per_s": np.polyfit(times, dist, 1)[0] * 1000})
par = pd.DataFrame(rows)
print(par.round(3).to_string(index=False))
for which in ("shortest", "longest"):
    s = par[par.ts == which]
    print(f"{which}: slopes > 0 in {int((s.slope_per_s > 0).sum())} of 8; median {s.slope_per_s.median():.2f} "
          f"per s; signed-rank p = {signed_rank(s.slope_per_s):.4f}")

**Result: reproduced, by the paper's test.** The signed-rank p is 0.109 for the shortest ts and 1.0
for the longest. For the shortest ts the distance to the middle trajectory grows in 6 of 8 (median
slope 75.8 per s), which the test over 8 does not find significant.

## alt-threshold-monotonic

> Aligned to the response, distances between trajectories for different ts decrease steadily up to
> movement initiation, without the converging then diverging pattern a ts-dependent threshold
> would produce.

Criterion: PCA per dataset on Go - 800 ms to Go, both priors, components explaining 60% of the
variance; per dataset, prior and non-middle ts (32 curves) the distance to the middle ts trajectory
at each 20 ms bin and the time of its minimum. Reproduced when the median time of the minimum is
in the last 40 ms before Go; partially when in the last 200 ms. A bin's time is its end (the last
bin ends at Go). The 75% threshold is shown as a variant.

In [ ]:
rows = []
for thr in (0.60, 0.75):
    for ds in DS:
        g = T[T.dataset == ds]
        tr = {(p, ts): rsg.go_rates(SP, g[(g.prior == p) & (g.ts == ts)]) for p in PRIORS for ts in rsg.PRIOR_TS[p]}
        pca, k = rsg.fit_pca(list(tr.values()), threshold=thr)
        Z = {key: pca.transform(v)[:, :k] for key, v in tr.items()}
        ends = np.arange(-780, 1, rsg.BIN_MS)
        for prior in PRIORS:
            tsv = rsg.PRIOR_TS[prior]
            for ts in tsv:
                if ts == tsv[2]:
                    continue
                dist = np.linalg.norm(Z[(prior, ts)] - Z[(prior, tsv[2])], axis=1)
                rows.append({"threshold": thr, "dataset": ds, "prior": prior, "ts": ts, "n_PC": k,
                             "t_min": ends[np.argmin(dist)], "dist_at_go": dist[-1], "dist_min": dist.min(),
                             "dist_at_-800": dist[0]})
thr_tab = pd.DataFrame(rows)
for thr, s in thr_tab.groupby("threshold"):
    print(f"\nPCs for {thr:.0%}:", s.groupby("dataset").n_PC.first().to_dict())
    print("Times of the minimum distance (ms before Go, bin end):", sorted(s.t_min.tolist()))
    print(f"Median time of the minimum: {s.t_min.median():.0f} ms; in the last 40 ms: {int((s.t_min >= -20).sum())}"
          f" of 32; in the last 200 ms: {int((s.t_min >= -180).sum())} of 32")
    print(f"Distance at Go above the minimum in {int((s.dist_at_go > s.dist_min + 1e-9).sum())} of 32; "
          f"median ratio of distance at Go to that at Go - 800 ms: {np.median(s.dist_at_go / s['dist_at_-800']):.2f}")

**Result: not reproduced.** The median time of the minimum distance is 230 ms before Go with 60% of
the variance (4 of 32 curves have it in the last 40 ms, 13 of 32 in the last 200 ms), and 200 ms
before Go with 75%. Distances shrink overall (the distance at Go is a median 0.43 of that at Go - 800
ms) but in 29 of 32 curves the distance at Go is above its minimum.

## gpfa-variance-inverted-u

> The variance across single trials of the GPFA neural state at Set projected on the encoding axis
> has an inverted-U profile over ts, lower at the extremes of each prior (one-tailed signed-rank
> test on quadratic coefficients, p = 0.0273, monkey H).

Criterion: GPFA (20 ms bins) per dataset on single trials from Ready to Set, both priors, the
selected units; latent dimensionality chosen by 3-fold cross-validated log-likelihood from 4 and 8;
orthonormalised latents. Per prior, u from the trial-averaged latents with attrition; each trial's
latent state in its last bin before Set projected on u; Xu z-scored over the prior's trials; the
variance at each ts; the quadratic coefficient of a polynomial fit of variance on ts. Reproduced
when a one-tailed signed-rank test (coefficient below zero) over the 8 dataset-priors has p < 0.05.

GPFA is elephant's implementation (Yu et al. 2009, the method the STAR Methods cite). Each trial
runs from `ready_time` for its nominal ts, so that its last bin ends at Set. EM runs for 200
iterations (elephant's default is 500), a standard choice made for time; the change of
log-likelihood over the last 10 iterations is printed. Folds are drawn with a fixed seed.

In [ ]:
import contextlib
import io

import neo
import quantities as pq
from elephant.gpfa import GPFA

logging.disable(logging.WARNING)  # elephant logs a warning per spike train it re-bins

rng_gpfa = np.random.default_rng(31)
EM_ITERS = 200


def spiketrains(t):
    out = []
    for r in t.itertuples():
        a, b = r.ready_time, r.ready_time + r.ts / 1000
        out.append([neo.SpikeTrain((s[(s >= a) & (s < b)] - a) * pq.s, t_start=0 * pq.s, t_stop=(r.ts / 1000) * pq.s)
                    for s in SP])
    return out


def quiet(fn, *args, **kw):
    """Run an elephant call without its progress prints and its re-binning warnings."""
    with warnings.catch_warnings(), contextlib.redirect_stdout(io.StringIO()):
        warnings.simplefilter("ignore")
        return fn(*args, **kw)


def gpfa_fit(trains, x_dim):
    np.random.seed(int(rng_gpfa.integers(1 << 31)))
    g = GPFA(bin_size=rsg.BIN_MS * pq.ms, x_dim=x_dim, em_max_iters=EM_ITERS, verbose=False)
    quiet(g.fit, trains)
    return g


t_g = time.time()
GP = {}
cv_rows = []
for ds in DS:
    g = T[T.dataset == ds].reset_index(drop=True)
    trains = spiketrains(g)
    folds = rng_gpfa.permutation(len(g)) % 3
    ll = {}
    for x_dim in (4, 8):
        ll[x_dim] = 0.0
        for f in range(3):
            m = gpfa_fit([trains[i] for i in np.flatnonzero(folds != f)], x_dim)
            ll[x_dim] += quiet(m.score, [trains[i] for i in np.flatnonzero(folds == f)])
    best = max(ll, key=ll.get)
    m = gpfa_fit(trains, best)
    lls = m.fit_info["log_likelihoods"]
    lat = quiet(m.transform, trains, returned_data=["latent_variable_orth"])
    GP[ds] = (g, lat)
    cv_rows.append({"dataset": ds, "trials": len(g), "cv_ll_4": ll[4], "cv_ll_8": ll[8], "x_dim": best,
                    "ll_change_last10": lls[-1] - lls[-11]})
print(pd.DataFrame(cv_rows).round(1).to_string(index=False))
print(f"GPFA time: {time.time() - t_g:.0f} s")

rows, GXU = [], {}
for ds in DS:
    g, lat = GP[ds]
    for prior in PRIORS:
        lo, hi = min(rsg.PRIOR_TS[prior]), max(rsg.PRIOR_TS[prior])
        ii = np.flatnonzero(g.prior.values == prior)
        nb = hi // rsg.BIN_MS
        acc, cnt = np.zeros((nb, lat[0].shape[0])), np.zeros(nb)
        for i in ii:
            L = lat[i].T
            acc[:len(L)] += L; cnt[:len(L)] += 1
        avg = acc / cnt[:, None]
        u = rsg.encoding_axis(avg, lo, hi)
        xu = np.array([lat[i][:, -1] @ u for i in ii])
        tsv = g.ts.values[ii]
        z = stats.zscore(xu)
        var = np.array([z[tsv == s].var(ddof=1) for s in rsg.PRIOR_TS[prior]])
        mean = np.array([xu[tsv == s].mean() for s in rsg.PRIOR_TS[prior]])
        GXU[(ds, prior)] = dict(var=var, mean=mean)
        rows.append({"dataset": ds, "prior": prior, "variance_by_ts": var.round(2),
                     "quad_coef": np.polyfit(np.array(rsg.PRIOR_TS[prior], float), var, 2)[0]})
gv = pd.DataFrame(rows)
print(gv.to_string(index=False, formatters={"quad_coef": "{:.3e}".format}))
print(f"\nQuadratic coefficient below zero in {int((gv.quad_coef < 0).sum())} of 8; one-tailed signed-rank "
      f"p = {signed_rank(gv.quad_coef, alternative='less'):.4f}")
print("Mean variance profile over the 8 (z-scored Xu):", np.mean([r for r in gv.variance_by_ts], axis=0).round(3))

**Result: not reproduced.** The quadratic coefficient is negative in 6 of 8 dataset-priors, and the
one-tailed signed-rank p is 0.074. The mean variance profile over the 8 is highest at the second ts
and lowest at the last two (0.80, 0.92, 0.88, 0.51, 0.44). Cross-validation chose 8 latent
dimensions in every dataset.

## gpfa-mean-sigmoid

> The mean of the single-trial GPFA projections is sigmoidal (signed-rank test on increments,
> p = 0.0078) and has a higher slope for the Short prior (one-tailed signed-rank, p = 0.0625).

Criterion: with the projections of gpfa-variance-inverted-u, not z-scored; the mean Xu at each ts;
increments as in beh-sigmoid-increments compared by a two-sided signed-rank test over the 8
(sigmoidal when middle exceeds extreme with p < 0.05); the slope of mean Xu on ts per dataset and
prior (higher for Short in all 4 datasets is the paper's p = 0.0625). Reproduced when both parts
hold; partially when one does.

In [ ]:
rows = []
for (ds, prior), e in GXU.items():
    inc = np.diff(e["mean"])
    rows.append({"dataset": ds, "prior": prior, "mean_xu": e["mean"].round(3), "middle_minus_extreme":
                 (inc[1] + inc[2]) / 2 - (inc[0] + inc[3]) / 2,
                 "slope": np.polyfit(np.array(rsg.PRIOR_TS[prior], float), e["mean"], 1)[0]})
gm = pd.DataFrame(rows)
print(gm.to_string(index=False, formatters={"middle_minus_extreme": "{:.4f}".format, "slope": "{:.5f}".format}))
print(f"\nMiddle > extreme in {int((gm.middle_minus_extreme > 0).sum())} of 8; two-sided signed-rank "
      f"p = {signed_rank(gm.middle_minus_extreme):.4f}")
gs = gm.pivot(index="dataset", columns="prior", values="slope")
print(gs.round(5).to_string())
print(f"Short slope above Long in {int((gs.Short > gs.Long).sum())} of 4; one-tailed signed-rank "
      f"p = {signed_rank(gs.Short - gs.Long, alternative='greater'):.4f}")

**Result: reproduced.** The middle increment of the mean projection exceeds the extreme one in 8 of 8
(two-sided signed-rank p = 0.0078), and the Short slope is above the Long slope in all 4 datasets
(one-tailed p = 0.0625, the paper's value).

## Blocked claims

- rnn-curved-manifold and rnn-perturbation: the recurrent networks, their training and their
  simulations are not in the dandiset, and the paper's code is available on request only, so
  neither the networks nor the paper's training procedure is public (`data_missing`).
- monkey-g: the dandiset holds one session of monkey H; no data of monkey G are in it or in any
  other dandiset found by searching the archive for the lab and the task (`data_missing`).

In [ ]:
print(f"Total run time: {(time.time() - T0) / 60:.1f} min; peak memory "
      f"{resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")

## Summary

| Claim | Outcome |
|---|---|
| beh-slopes | not reproduced: 6 of 8 slopes in (0, 1); the eye Short slopes are 1.011 and 1.110 |
| beh-overlap-prior | not reproduced: both median biases positive (+28.3, +47.6 ms), p = 0.073 |
| beh-sigmoid-increments | reproduced: 7 of 8, p = 0.0156 |
| beh-long-more-bias | reproduced: 4 of 4 |
| beh-bayes-model | partially reproduced: bias R2 >= 0.8 in 2 of 4, SD correlation positive in 4 of 4 |
| glm-prior-ts | not reproduced: 79.2% prior-, 45.8% ts-dependent |
| dfr-overlap-support | reproduced: peak at 480 to 640 ms after Ready, above the shuffle null |
| pca-dimensionality | not reproduced: 1 of 8 in range (2 PCs reach 75% in the estimation epoch) |
| encoding-r2 | reproduced: Hand Left 0.994 and 0.986 |
| encoding-bayes-beats-linear | not reproduced: Bayesian better in 3 of 4 for each prior |
| short-in-long-control | reproduced by the paper's test (p = 0.25, n = 4) |
| neural-long-more-bias | reproduced: 4 of 4 |
| speed-vs-projection | partially reproduced: r = -0.926 and -0.834, outside tolerance |
| initial-speed-biased | reproduced: 7 of 8, p = 0.039 |
| speed-predicts-tp | partially reproduced: r = -0.720 and -0.699, outside tolerance |
| alt-speed-unmodulated | reproduced by the paper's test (p = 0.125, n = 4); speed rises in 8 of 8 |
| alt-transient-parallel | reproduced by the paper's test (p = 0.109 and 1.0) |
| alt-threshold-monotonic | not reproduced: median minimum 230 ms before Go |
| gpfa-variance-inverted-u | not reproduced: 6 of 8 negative, p = 0.074 |
| gpfa-mean-sigmoid | reproduced: p = 0.0078 and 0.0625 |
| rnn-curved-manifold | blocked: data missing |
| rnn-perturbation | blocked: data missing |
| monkey-g | blocked: data missing |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

How the analysis differs from the paper's. It uses one session of monkey H, where the paper pools
17 sessions of H and 12 of G, so every test over "datasets" has 4 in place of 8, and tests the paper
ran over sessions run here over the 4 datasets of one session. Firing rates are not normalised
(claims header), which leaves 2 components at 75% in most datasets and makes the estimation PC
space 2-dimensional. The paper's code is not public; method details come from the claims file and
the STAR Methods, as cited in each section. GPFA runs 200 EM iterations, fewer than elephant's
default of 500.

What was found about the dataset. It holds one of the paper's 29 sessions: 1160 trials of the
train file (983 completed and not outliers) and 54 units, 48 of which pass the paper's selection,
the number the paper gives for the GPFA session of monkey H. The 283 trials of the test file have no
task variables, as the benchmark withholds them. The `iti` description reverses the two priors; `tp`
exceeds `go_time - set_time` by a median of 17.9 ms without explanation; units carry no electrode,
probe or single/multi-unit label.